<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 4 (Jueves) — No supervisado — Agrupación (K-means) y asociación (Apriori)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🛒 K-means: segmentar a los clientes de un almacén mayorista

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Segmentar a los clientes mayoristas según su consumo para preparar promociones y planear la distribución. |
| 🔍 Comprensión de los datos | Cargamos 440 clientes con 8 variables (canal, región y gasto anual en 6 categorías de producto). |
| 🧹 Preparación de los datos | Elegimos las 6 variables de gasto; sin estandarizar en el flujo principal (el refuerzo muestra el efecto de la escala). |
| 🤖 Modelado | Ajustamos K-means con k = 5 y elegimos k con el método del codo. |
| 📊 Evaluación | Analizamos el tamaño de los grupos (Pareto) y su perfil medio de consumo. |
| 🚀 Despliegue y conclusiones | Describimos cada segmento y proponemos acciones comerciales. |

> 🎯 **Analogía:** un gerente que acomoda a sus clientes en "mesas" según lo que compran: quienes llenan el carrito de frescos, quienes compran detergentes y lácteos, quienes compran poco de todo.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 👉 **4** | Jueves | **No supervisado: agrupación y asociación** |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('ventas.csv', 'dia_5')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Segmentar a los clientes mayoristas según su consumo para preparar promociones y planear la distribución.

📚 Lectura recomendada: [3] [8]

 # Algoritmos de clasificación **NO supervisada**
Un algoritmo de agrupamiento (en inglés, clustering) es un procedimiento de agrupación de una serie de vectores de acuerdo con un criterio. Esos criterios son por lo general distancia o similitud. La cercanía se define en términos de una determinada función de distancia, como la euclídea, aunque existen otras más robustas o que permiten extenderla a variables discretas. La medida más utilizada para medir la similitud entre los casos es la matriz de correlación entre los nxn casos. Sin embargo, también existen muchos algoritmos que se basan en la maximización de una propiedad estadística llamada verosimilitud.

### 🧪 Teoría rápida: ¿cómo agrupa K-means?

En el aprendizaje **no supervisado** no hay etiqueta que predecir: se buscan **estructuras** en los datos. K-means reparte las observaciones en *k* grupos alternando dos pasos hasta que los centroides dejan de moverse: (1) asignar cada punto al centroide más cercano y (2) recalcular cada centroide como la media de su grupo [3] [4].

![Datos sin etiquetas](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_01.png)
*Figura: datos sin etiquetas. [12]*

![K-means con k = 3](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_02.png)
*Figura: K-means con k = 3; las estrellas rojas son los centroides. [12]*

**¿Cómo elegir k?** El **método del codo** busca el *k* a partir del cual la suma de cuadrados dentro de los grupos (SSE) deja de bajar bruscamente [6]; el **coeficiente de silueta** mide qué tan bien separado queda cada punto de los demás grupos [7].

![Método del codo](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_03.png)
*Figura: método del codo (el codo está en k = 3). [12]*

⚠️ K-means usa **distancias**, por lo que las variables con valores grandes dominan el resultado; normalmente se **estandarizan** (y, si son muy asimétricas, se aplica logaritmo) [8].

**K-means** es un algoritmo de clustering no supervisado que se utiliza para agrupar datos en k grupos diferentes basados en similitudes en sus características. El objetivo es minimizar la varianza intra-cluster y maximizar la varianza inter-cluster.

El algoritmo K-means comienza seleccionando aleatoriamente k centroides de los datos. Luego, para cada punto de datos, se encuentra el centroide más cercano y se le asigna a ese grupo. Después de que todos los puntos de datos hayan sido asignados a un grupo, se recalculan los centroides como la media de los puntos de datos en cada grupo. Este proceso se repite hasta que los centroides ya no cambian significativamente o se alcanza el número máximo de iteraciones.

En resumen, los pasos del algoritmo K-means son:



*   Seleccione aleatoriamente k centroides de los datos
*   Asigne cada punto de datos al centroide más cercano
*   Recalcule los centroides como la media de los puntos de datos en cada grupo
*   Repita los pasos 2 y 3 hasta que se alcance un criterio de parada (por ejemplo, los centroides no cambian significativamente) o se alcance el número máximo de iteraciones.

Una de las ventajas del algoritmo K-means es su simplicidad y rapidez de ejecución. Sin embargo, su desventaja es que puede ser sensible a los centroides iniciales seleccionados aleatoriamente y puede tener dificultades para manejar grupos de formas irregulares o tamaños desiguales

## Análisis del Problema
El conjunto de datos corresponde a las ventas de un almacén de ventas al por mayor. Analizar el patrón de compras de sus clientes, le permitiría al negocio segmentar sus clientes y preparar promociones o eventos especiales para cada grupo de clientes.

La ventaja que obtendría el negocio al hacer algo así puede verse reflejada en un incremente en las ventas de los productos que caracterizan a cada uno de los clientes. Asimismo, el negocio podría planear mejor la forma en que distribuye su producto, pues si se encuentra un patrón de consumo alto de un producto que es bajo en las otras zonas o grupos, se puede tomar la decisión de enviar mayores cantidades de ese producto a donde hay mayor consumo.

### Librerias

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import plotly.graph_objs as go

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos 440 clientes con 8 variables (canal, región y gasto anual en 6 categorías de producto).

📚 Lectura recomendada: [9]

## Entendimiento de los Datos

El conjunto de datos contiene las siguientes variables:

- Channel: canal de venta (1 = Horeca - 298 observaciones, 2 = Retail  - 142 observaciones).
- Region: región en la cual se hizo la venta (1 = Lisboa - 77 observaciones, 2 = Oporto - 47 observaciones, 3 = Otra Región - 316 observaciones).
- Fresh: cantidad de dinero en producto fresco.
- Grocery: cantidad de dinero en comestibles.
- Frozen: cantidad de dinero en alimento congelado.
- Detergents_Paper: cantidad de dinero en detergente.
- Delicassen: cantidad de dinero en delicatessen.

## Exploración de los Datos

In [ ]:
# Cargar el archivo CSV en un DataFrame llamado "ventas"
ventas = pd.read_csv('sample_data/ventas.csv')

# Convertir las columnas Region y Channel en factores
ventas['Region'] = pd.Categorical(ventas['Region'])
ventas['Channel'] = pd.Categorical(ventas['Channel'])

# Convertir el data frame de ventas a formato "largo"
ventas_largo = pd.melt(ventas, id_vars=['Channel'], value_vars=['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen'], var_name='Producto', value_name='Ventas')

# Ver la estructura del conjunto de datos utilizando la función "info()"
ventas.info()

In [ ]:
ventas

---
## 🔍 Fase 2 · Comprensión de los datos — exploración de canal y región

Cargamos 440 clientes con 8 variables (canal, región y gasto anual en 6 categorías de producto).

La mayoría de las ventas se registraron en la región #3 (otra región), mientras que la mayoría de las  ventas se hicieron por medio del canal # 1 (Horeca), tal como se puede apreciar en los siguientes gráficos:

In [ ]:
# Gráfico de barras para la distribución de ventas por canal
ventas['Channel'].value_counts().plot(kind='bar')
plt.title('Distribución de ventas por canal')
plt.xlabel('Canal')
plt.ylabel('Cantidad de ventas')
plt.show()

# Gráfico de barras para la distribución de ventas por región
ventas['Region'].value_counts().plot(kind='bar')
plt.title('Distribución de ventas por región')
plt.xlabel('Región')
plt.ylabel('Cantidad de ventas')
plt.show()

In [ ]:
# Crear el boxplot con seaborn
sns.set(style='whitegrid')
plt.figure(figsize=(8,6))
sns.boxplot(x='Producto', y='Ventas', hue='Channel', data=ventas_largo, palette='Set1')
plt.xlabel('Producto')
plt.ylabel('Ventas')
plt.title('Ventas por producto y por canal')
plt.show()

---
## 🧹 Fase 3 · Preparación de los datos — seleccionar las variables de gasto

Elegimos las 6 variables de gasto; sin estandarizar en el flujo principal (el refuerzo muestra el efecto de la escala).

📚 Lectura recomendada: [8]

## Modelo de IA

Para agrupar a los clientes con base al consumo, vamos a utilizar solo las variables Fresh, Milk, Grocery, Frozen, Detergents_Paper y Delicassen. En este caso, vamos a intentar agrupar los clientes en 5 grupos diferentes.

---
## 🤖 Fase 4 · Modelado

Ajustamos K-means con k = 5 y elegimos k con el método del codo.

📚 Lectura recomendada: [4] [5] [6]

In [ ]:
# Realizar el clustering con k-means (k = 5; random_state y n_init fijos para que el resultado sea reproducible)
modelo_agrupacion = KMeans(n_clusters=5, random_state=42, n_init=10).fit(ventas.iloc[:, 2:8])

# Asignar etiquetas a cada observación
etiquetas = modelo_agrupacion.labels_

# Asignar las etiquetas como una nueva columna en el DataFrame ventas
ventas['Grupo'] = etiquetas

In [ ]:
ventas

In [ ]:
# Obtener los centroides del modelo
centroides = modelo_agrupacion.cluster_centers_

# Crear un gráfico de dispersión con los centroides
plt.scatter(ventas.iloc[:, 3], ventas.iloc[:, 4], c=etiquetas, cmap='viridis')
plt.scatter(centroides[:, 1], centroides[:, 2], marker='*', s=300, c='red')   # Milk y Grocery (columnas 1 y 2 de los centroides)
plt.xlabel('Milk')
plt.ylabel('Grocery')
plt.title('Agrupación de ventas')
plt.show()

En este ejemplo, estamos visualizando los centroides en un gráfico de dispersión utilizando las variables Milk y Grocery del DataFrame ventas. Primero, obtenemos los centroides utilizando la propiedad cluster_centers_ del objeto KMeans. Luego, utilizamos plt.scatter para crear un gráfico de dispersión con los datos de ventas, utilizando las etiquetas asignadas a cada observación para colorear los puntos. Finalmente, utilizamos plt.scatter de nuevo para añadir los centroides en el gráfico, utilizando el marcador * y el color rojo.

In [ ]:
# Obtener los centroides del modelo
centroides = modelo_agrupacion.cluster_centers_

# Transformar los centroides en un DataFrame
centroides_df = pd.DataFrame(centroides, columns=ventas.columns[2:8])

# Mostrar los centroides en formato de tabla
print(centroides_df)

---
## 🤖 Fase 4 · Modelado — elegir k: método del codo

Ajustamos K-means con k = 5 y elegimos k con el método del codo.

### Obtener el valor K

In [ ]:
# calcular el SSE (inercia: suma de cuadrados dentro de los grupos) para diferentes valores de k
sse = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(ventas.iloc[:, 2:8])
    sse.append(km.inertia_)

In [ ]:
# trazar gráfico de SSE vs. k
plt.plot(k_range, sse)
plt.xlabel('Número de clusters')
plt.ylabel('SSE (inercia)')
plt.title('Método del codo')
plt.show()

---
## 📊 Fase 5 · Evaluación

Analizamos el tamaño de los grupos (Pareto) y su perfil medio de consumo.

📚 Lectura recomendada: [7]

In [ ]:
# Reetiquetar los grupos del más grande (1) al más pequeño (5) para que la descripción sea estable entre ejecuciones
orden = ventas['Grupo'].value_counts().index
ventas['Grupo'] = ventas['Grupo'].map({g: i + 1 for i, g in enumerate(orden)})
ventas['Grupo'].value_counts().sort_index()

## Evaluación

Al hacer un análisis de Pareto sobre los grupos creados (ordenados del más grande al más pequeño) se aprecia que los grupos 1 y 2 reúnen cerca del **77 %** de los clientes y los tres primeros el **95 %**. Los grupos grandes representan al cliente promedio; los pequeños (4 y 5) son clientes poco frecuentes pero de gasto muy alto, que conviene atender de forma personalizada.

In [ ]:
# Crear una tabla de frecuencia de la variable "cluster"
cluster_freq = ventas["Grupo"].value_counts()

# Calcular el porcentaje acumulado de las observaciones por grupo
cumulative_perc = 100 * cluster_freq.cumsum() / cluster_freq.sum()

# Crear un gráfico de barras para las frecuencias de cluster
fig, ax1 = plt.subplots()

ax1.bar(cluster_freq.index, cluster_freq.values, color='b')
ax1.set_xlabel('Cluster')
ax1.set_ylabel('Frecuencia', color='b')
ax1.tick_params('y', colors='b')

# Crear un gráfico de línea para el porcentaje acumulado
ax2 = ax1.twinx()

ax2.plot(cluster_freq.index, cumulative_perc, '-ro', linewidth=2, markersize=7)
ax2.set_ylabel('Porcentaje acumulado (%)', color='r')
ax2.tick_params('y', colors='r')

# Establecer el título del gráfico
plt.title('Distribución de Pareto de los Clusters')

# Mostrar el gráfico
plt.show()

In [ ]:
# Crear una tabla de frecuencia de la variable "cluster"
cluster_freq = ventas["Grupo"].value_counts()

# Crear un gráfico de barras de la distribución de frecuencia
plt.bar(cluster_freq.index, cluster_freq.values)

# Establecer el título del gráfico y las etiquetas de los ejes
plt.title("Distribución de frecuencia de los clusters")
plt.xlabel("Cluster")
plt.ylabel("Frecuencia")

# Mostrar el gráfico
plt.show()

In [ ]:
# Crear una tabla de frecuencia de la variable "cluster"
cluster_freq = ventas["Grupo"].value_counts()

# Crear un dataframe con la tabla de frecuencia y el porcentaje acumulado
cluster_freq_df = pd.DataFrame({'Cluster': cluster_freq.index, 'Frecuencia': cluster_freq.values})
cluster_freq_df['Porcentaje acumulado'] = cluster_freq_df['Frecuencia'].cumsum() / cluster_freq_df['Frecuencia'].sum() * 100

# Mostrar la tabla de frecuencia con el porcentaje acumulado
print(cluster_freq_df)

# Evaluación

In [ ]:
ventas.groupby('Grupo')[['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']].mean()


In [ ]:
# Obtener el valor máximo de cada grupo para todas las columnas
ventas.groupby('Grupo')[['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']].max()

In [ ]:
# Obtener el valor mínimo de cada grupo para todas las columnas
ventas.groupby('Grupo')[['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']].min()

In [ ]:
# Obtener tanto el valor máximo como el mínimo de cada grupo para todas las columnas
ventas.groupby('Grupo')[['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']].agg(['max', 'min'])

---
## 🚀 Fase 6 · Despliegue y conclusiones — interpretación de los segmentos

Describimos cada segmento y proponemos acciones comerciales.

📚 Lectura recomendada: [10]

Perfil de los grupos (resultado con `random_state=42`; los números corresponden al gasto medio anual; el promedio general es Fresh ≈ 12 000, Milk ≈ 5 800, Grocery ≈ 8 000, Frozen ≈ 3 100, Detergents_Paper ≈ 2 900, Delicassen ≈ 1 500):

- **Grupo 1 (233 clientes, 53 %)** — mayormente canal 1 (Horeca, 84 %). Gasto **bajo** en todas las categorías (Fresh ≈ 6 400). Es el cliente pequeño o promedio.
- **Grupo 2 (106 clientes, 24 %)** — canal 1 (Horeca, 78 %). Gasto **alto en productos frescos** (≈ 23 700, el doble del promedio) y moderado en el resto.
- **Grupo 3 (81 clientes, 18 %)** — canal 2 (Retail, 91 %). Gasto **alto en leche, comestibles y detergentes** (Milk ≈ 12 100, Grocery ≈ 18 500, Detergents_Paper ≈ 8 100) y bajo en frescos.
- **Grupo 4 (13 clientes, 3 %)** — 100 % Horeca. Gasto **muy alto en frescos** (≈ 56 500), congelados (≈ 15 500) y delicatessen (≈ 7 200).
- **Grupo 5 (7 clientes, 1.6 %)** — 100 % Retail. Gasto **muy alto en leche, comestibles y detergentes** (Milk ≈ 38 100, Grocery ≈ 56 100, Detergents_Paper ≈ 27 600).

La **región** casi no distingue los grupos (alrededor del 70 % de cada grupo está en la región 3), de modo que el patrón de consumo depende sobre todo del **canal** (Horeca vs. Retail).

In [ ]:
ventas

---
## 🧪 Refuerzo: K-means desde cero y el efecto de la escala de las variables

**Parte 1 — Programar K-means.** El algoritmo de Lloyd cabe en pocas líneas: asignar → actualizar → repetir [4]. Como el resultado depende de los centroides iniciales (puede caer en un mínimo local) [5], se repite con varias semillas y se conserva la mejor (menor SSE), que es lo que hace `n_init` en `scikit-learn`.

**Parte 2 — Escala.** Las ventas en *Fresh* y *Grocery* llegan a decenas de miles, mientras que *Delicassen* ronda los miles; sin transformar, K-means forma grupos dominados por unos pocos clientes enormes (grupos de 2, 5 o 7 clientes). Comparamos los datos crudos con los datos en escala logarítmica y estandarizados usando el **coeficiente de silueta** [7].

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

columnas = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X_crudo = ventas[columnas]
X_log = StandardScaler().fit_transform(np.log1p(X_crudo))       # log (corrige la asimetría) + estandarizar

# --- Parte 1: K-means desde cero (algoritmo de Lloyd) ---
def kmeans_desde_cero(X, k, iteraciones=100, semilla=0):
    rng = np.random.default_rng(semilla)
    centros = X[rng.choice(len(X), k, replace=False)]
    for _ in range(iteraciones):
        dist = np.linalg.norm(X[:, None, :] - centros[None, :, :], axis=2)     # distancia de cada punto a cada centro
        etiquetas = dist.argmin(axis=1)                                          # 1) asignar al centro más cercano
        nuevos = np.array([X[etiquetas == j].mean(axis=0) if np.any(etiquetas == j) else centros[j] for j in range(k)])
        if np.allclose(nuevos, centros): break
        centros = nuevos                                                         # 2) mover cada centro a la media de su grupo
    return etiquetas, centros

mejor = min((kmeans_desde_cero(X_log, 4, semilla=s) for s in range(10)), key=lambda r: ((X_log - r[1][r[0]]) ** 2).sum())
sk = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_log)
print('SSE (nuestro K-means, mejor de 10 semillas):', round(((X_log - mejor[1][mejor[0]]) ** 2).sum(), 1))
print('SSE (scikit-learn, n_init=10)              :', round(sk.inertia_, 1))
print('Coincidencia entre ambas agrupaciones (ARI) :', round(adjusted_rand_score(mejor[0], sk.labels_), 3))

# --- Parte 2: datos crudos vs. log + estandarizado ---
filas = []
for k in range(2, 9):
    for nombre, Z in (('crudos', X_crudo), ('log + estandarizado', X_log)):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Z)
        filas.append({'k': k, 'datos': nombre, 'silueta': round(silhouette_score(Z, km.labels_), 3), 'grupo más pequeño': int(np.bincount(km.labels_).min())})
pd.DataFrame(filas).pivot(index='k', columns='datos', values=['silueta', 'grupo más pequeño'])

### ✍️ Ejercicios
1. ¿Con cuál escala son más equilibrados los tamaños de los grupos? ¿Por qué la silueta de los datos crudos es más alta aunque los grupos sean menos útiles?
2. Repita el modelo principal (k = 5) con `X_log` y describa con palabras cada grupo: ¿encuentra el mismo grupo de clientes *Retail* con mucho detergente?
3. ¿Qué acción comercial propondría para el grupo de clientes con gasto altísimo en productos frescos (Horeca)?

---
## ✅ Conclusiones y siguientes pasos

K-means convirtió 440 clientes en **cinco segmentos accionables**: un gran bloque de clientes pequeños, un grupo orientado a frescos, un grupo *Retail* de lácteos/comestibles/detergentes y dos grupos pequeños de gasto muy alto que justifican atención personalizada. Recuerde que K-means depende de la escala y de los centroides iniciales [8]: valide la estabilidad con varias semillas y pruebe variables transformadas.

---
## 📚 Referencias

Fuentes adicionales para profundizar: [11].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** MacQueen, J. (1967). Some methods for classification and analysis of multivariate observations. *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability, 1*, 281–297.

**[4]** Lloyd, S. P. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory, 28*(2), 129–137. https://doi.org/10.1109/TIT.1982.1056489

**[5]** Arthur, D., & Vassilvitskii, S. (2007). k-means++: The advantages of careful seeding. *Proceedings of the 18th Annual ACM-SIAM Symposium on Discrete Algorithms (SODA '07)*, 1027–1035.

**[6]** Thorndike, R. L. (1953). Who belongs in the family? *Psychometrika, 18*(4), 267–276. https://doi.org/10.1007/BF02289263

**[7]** Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics, 20*, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7

**[8]** Jain, A. K. (2010). Data clustering: 50 years beyond K-means. *Pattern Recognition Letters, 31*(8), 651–666. https://doi.org/10.1016/j.patrec.2009.09.011

**[9]** Cardoso, M. (2013). *Wholesale customers* [Conjunto de datos]. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/292/wholesale+customers

**[10]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[11]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[12]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)